---

### Cell 1: 設定裝置與實驗超參數


In [ ]:
# ✅ 匯入核心模組
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import DataLoader, Dataset, ConcatDataset
import numpy as np
import os
import zipfile
from PIL import Image
from tqdm.notebook import tqdm  # 用於訓練進度條

# ✅ 運算裝置設定
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Using device: {DEVICE}")

# ✅ 隨機種子（確保結果可重現）
torch.manual_seed(42)
np.random.seed(42)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(42)

# ✅ PACS 基本資訊
PACS_DOMAINS = ['art_painting', 'cartoon', 'photo', 'sketch']
PACS_CATEGORIES = ['dog', 'elephant', 'giraffe', 'guitar', 'horse', 'house', 'person']

# ✅ 實驗超參數
MODEL_CHOICE = 'resnet18'
PRETRAINED = True
NUM_EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 3e-5

# ✅ 實驗設定摘要
print(f"🔧 Model: {MODEL_CHOICE}, Epochs: {NUM_EPOCHS}, Batch: {BATCH_SIZE}, LR: {LEARNING_RATE}")


---

### Cell 2: 掛載 Google Drive 並自動下載與解壓 PACS 資料集



In [ ]:
# ✅ 匯入必要模組
import os
import zipfile
import shutil
from google.colab import drive

# ✅ 掛載 Google Drive
drive.mount('/content/drive', force_remount=True)

# ✅ 創建 Google Drive 資料夾
drive_root = "/content/drive/MyDrive/domainbed_project"
os.makedirs(drive_root, exist_ok=True)

# ✅ 上傳 kaggle.json（第一次用這個帳號）
# 👉 請點左側「📁檔案」→ 右鍵上傳 kaggle.json 到 /content
kaggle_json_path = "/content/kaggle.json"
kaggle_config_dir = "/root/.kaggle"
os.makedirs(kaggle_config_dir, exist_ok=True)
shutil.copy(kaggle_json_path, os.path.join(kaggle_config_dir, "kaggle.json"))
os.chmod(os.path.join(kaggle_config_dir, "kaggle.json"), 0o600)
print("✅ kaggle.json 已成功設置")

# ✅ 安裝 kaggle CLI（如尚未安裝）
!pip install -q kaggle

# ✅ 從 Kaggle 下載 PACS 資料集（如尚未存在）
zip_path_drive = os.path.join(drive_root, "pacs-dataset.zip")
if not os.path.exists(zip_path_drive):
    print("⬇️ 正在從 Kaggle 下載 PACS 資料集...")
    !kaggle datasets download -d ma3ple/pacs-dataset -p "{drive_root}"
else:
    print("✅ 已存在 pacs-dataset.zip，跳過下載")

# ✅ 解壓縮 zip 到本地 /content/kfold
local_zip_path = "/content/pacs-dataset.zip"
shutil.copy(zip_path_drive, local_zip_path)

with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content")
print("✅ 解壓完成，原始資料夾名稱為 'kfold'")

# ✅ 重新命名資料夾為 'PACS'
original_folder = "/content/kfold"
final_folder = "/content/PACS"
if os.path.exists(final_folder):
    shutil.rmtree(final_folder)
if os.path.exists(original_folder):
    os.rename(original_folder, final_folder)
    print("✅ 'kfold' 已重新命名為 'PACS'")
else:
    print("❌ 解壓後找不到 'kfold'，請檢查 zip 壓縮內容")

# ✅ 設定資料根路徑
BASE_PACS_PATH = final_folder
print(f"📂 Dataset base path is configured as: '{BASE_PACS_PATH}'")

---

### Cell 3: 資料集定義與 TTA 圖像轉換設定





In [ ]:
from pathlib import Path
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import os

# ✅ PACS 單圖版 Dataset（非 TTA 用）
class PACS_Dataset(Dataset):
    def __init__(self, base_path, domain_name, categories, transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []
        for label_idx, category in enumerate(categories):
            category_path = os.path.join(base_path, domain_name, category)
            if not os.path.isdir(category_path):
                continue
            for img_file in os.listdir(category_path):
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.image_paths.append(os.path.join(category_path, img_file))
                    self.labels.append(label_idx)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

# ✅ 圖像標準化參數
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# ✅ 標準訓練用轉換（不含 TTA）
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# ✅ TTA 增強版本轉換
tta_transforms = [
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomRotation(degrees=15),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
]

# ✅ 驗證與測試共用轉換（不加增強）
val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

---

### Cell 4: 模型構建與驗證函數定義

In [ ]:
# 📦 Cell 4: 模型構建與驗證函數定義

import torch.nn as nn
from torchvision import models

# ✅ 構建 ResNet 模型
def get_model(num_classes=len(PACS_CATEGORIES), pretrained=True):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    return model

# ✅ 測試模型建立
try:
    test_model = get_model(num_classes=len(PACS_CATEGORIES))
    print("✅ Model created successfully.")
except Exception as e:
    print(f"❌ Error creating model: {e}")

# ✅ 基本驗證函數（非 TTA）
def eval_one_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    epoch_loss = running_loss / total if total > 0 else 0.0
    epoch_acc = correct / total if total > 0 else 0.0
    return epoch_loss, epoch_acc


---

### Cell 5: 訓練與 TTA 擴增資料集定義

In [ ]:
# 📦 Cell 5: 訓練邏輯與 TTA 擴增資料集定義

from torch.utils.data import Dataset
from PIL import Image
from tqdm.notebook import tqdm

# ✅ 標準訓練函數
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, labels in tqdm(dataloader, desc="Train", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

# ✅ TTA 驗證（eval_one_epoch_tta）
def eval_one_epoch_tta(model, dataloader, criterion, device, tta_transforms, epoch_desc="Evaluating (TTA)"):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    progress_bar = tqdm(dataloader, desc=epoch_desc, leave=False)

    with torch.no_grad():
        for inputs_pil_list, labels in progress_bar:
            labels = labels.to(device)
            batch_outputs_for_tta_versions = []

            for t_transform in tta_transforms:
                current_tta_batch_tensors = torch.stack([t_transform(pil_img) for pil_img in inputs_pil_list])
                current_tta_batch_tensors = current_tta_batch_tensors.to(device)
                outputs = model(current_tta_batch_tensors)
                batch_outputs_for_tta_versions.append(outputs)

            avg_logits = torch.stack(batch_outputs_for_tta_versions).mean(dim=0)
            loss = criterion(avg_logits, labels)

            running_loss += loss.item() * labels.size(0)
            probs = torch.softmax(avg_logits, dim=1)
            preds = probs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    epoch_loss = running_loss / total if total > 0 else 0.0
    epoch_acc = correct / total if total > 0 else 0.0
    return epoch_loss, epoch_acc

# ✅ TTA 多版本 Dataset（每張圖多種增強）
class MultiAugmentPACS_Dataset(Dataset):
    def __init__(self, base_path, domain_name, categories, transform_list):
        self.transform_list = transform_list
        self.image_paths = []
        self.labels = []
        for label_idx, category in enumerate(categories):
            category_path = os.path.join(base_path, domain_name, category)
            if not os.path.isdir(category_path):
                continue
            for img_file in os.listdir(category_path):
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.image_paths.append(os.path.join(category_path, img_file))
                    self.labels.append(label_idx)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        imgs = [t(img) for t in self.transform_list]
        return imgs, label

# ✅ 多版本圖像的訓練函數
def train_one_epoch_multi_aug(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs_list, labels in tqdm(dataloader, desc="Train", leave=False):
        labels = labels.to(device)
        loss_list = []
        optimizer.zero_grad()
        for imgs in imgs_list:
            outputs = model(imgs.to(device))
            loss = criterion(outputs, labels)
            loss_list.append(loss)
        final_loss = sum(loss_list) / len(loss_list)
        final_loss.backward()
        optimizer.step()

        running_loss += final_loss.item() * labels.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


---

### Cell 6: Training with Multi-Domain TTA Evaluation and Resume Support

In [ ]:
import json
import os
import numpy as np
import torch
from torch.utils.data import DataLoader

# === 結果資料夾與路徑設定 ===
RESULTS_DIR = "/content/drive/MyDrive/xAi_DG_TTA/lodo_tta_models"
os.makedirs(RESULTS_DIR, exist_ok=True)
RESUME_PATH = os.path.join(RESULTS_DIR, "checkpoint_latest.pth")
LOG_JSON_PATH = os.path.join(RESULTS_DIR, "train_log.json")

# 用來記錄每個domain最佳準確率和模型路徑
best_acc_per_domain = {}
best_model_paths = {}

# 預設斷點紀錄格式
train_log = {
    "completed_epochs": {},  # 形如 {"art_painting": last_epoch_num, ...}
    "metrics": {}           # 形如 {"art_painting": [epoch_metrics], ...}
}

# 檢查是否有斷點檔，有的話讀取斷點
if os.path.exists(RESUME_PATH):
    print("🔁 檢測到暫存模型，正在恢復訓練...")
    checkpoint = torch.load(RESUME_PATH)
    best_acc_per_domain = checkpoint.get("best_acc_per_domain", {})
    best_model_paths = checkpoint.get("best_model_paths", {})
    train_log = checkpoint.get("train_log", train_log)
else:
    # 初始化
    for d in PACS_DOMAINS:
        best_acc_per_domain[d] = 0.0
        best_model_paths[d] = None
        train_log["completed_epochs"][d] = 0
        train_log["metrics"][d] = []

# 對每個 domain 執行 Leave-One-Domain-Out 訓練
for target_domain in PACS_DOMAINS:
    print(f"\n===== Leave-One-Domain-Out training for domain: {target_domain} =====")

    # 判斷該 domain 已完成多少 epoch
    start_epoch = train_log["completed_epochs"].get(target_domain, 0)
    if start_epoch >= NUM_EPOCHS:
        print(f"Domain {target_domain} 已完成 {NUM_EPOCHS} 個 epochs，跳過。")
        continue

    # 訓練用 domains（除去 target_domain）
    source_domains = [d for d in PACS_DOMAINS if d != target_domain]
    train_datasets = [
        MultiAugmentPACS_Dataset(BASE_PACS_PATH, d, PACS_CATEGORIES, transform_list=tta_transforms)
        for d in source_domains
    ]
    train_dataset = torch.utils.data.ConcatDataset(train_datasets)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

    # 驗證集是留出 domain
    val_dataset = PACS_Dataset(BASE_PACS_PATH, target_domain, PACS_CATEGORIES, transform=val_test_transform)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = get_model(len(PACS_CATEGORIES), PRETRAINED).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    criterion = torch.nn.CrossEntropyLoss()

    # 如果有保存最佳模型，載入
    best_model_path = os.path.join(RESULTS_DIR, f"best_model_{target_domain}_tta.pth")
    if os.path.exists(best_model_path):
        model.load_state_dict(torch.load(best_model_path))
        print(f"✅ 載入 {target_domain} 最佳模型參數。")

    # 從斷點繼續訓練
    for epoch in range(start_epoch, NUM_EPOCHS):
        print(f"\n[Domain {target_domain}] Epoch {epoch+1}/{NUM_EPOCHS}")
        train_loss, train_acc = train_one_epoch_multi_aug(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc = eval_one_epoch(model, val_loader, criterion, DEVICE)
        print(f"Train Acc: {train_acc:.3f} | Val Acc: {val_acc:.3f}")

        # 記錄 log
        train_log["metrics"][target_domain].append({
            "epoch": epoch + 1,
            "train_acc": train_acc,
            "val_acc": val_acc,
            "train_loss": train_loss,
            "val_loss": val_loss
        })

        # 保存最佳模型
        if val_acc > best_acc_per_domain[target_domain]:
            best_acc_per_domain[target_domain] = val_acc
            torch.save(model.state_dict(), best_model_path)
            best_model_paths[target_domain] = best_model_path
            print(f"⭐ 新最佳模型，儲存到: {best_model_path}")

        # 更新完成 epoch
        train_log["completed_epochs"][target_domain] = epoch + 1

        # 寫出斷點檔
        torch.save({
            "best_acc_per_domain": best_acc_per_domain,
            "best_model_paths": best_model_paths,
            "train_log": train_log,
        }, RESUME_PATH)

    print(f"Domain {target_domain} 訓練完成。")

# 訓練完成後，列印並存檔整體結果
print("\n=== Leave-One-Domain-Out (LODO) TTA 訓練結果總結 ===")
for domain in PACS_DOMAINS:
    print(f"{domain} 最佳 Val Acc: {best_acc_per_domain[domain]:.3f}")

avg_acc = np.mean(list(best_acc_per_domain.values()))
worst_acc = np.min(list(best_acc_per_domain.values()))
print(f"平均準確率: {avg_acc:.3f}, 最差準確率: {worst_acc:.3f}")

# 儲存結果 JSON
result_path = os.path.join(RESULTS_DIR, "lodo_tta_results.json")
with open(result_path, "w") as f:
    json.dump({
        "type": "lodo_tta",
        "domains": PACS_DOMAINS,
        "best_acc_per_domain": best_acc_per_domain,
        "average_acc": avg_acc,
        "worst_acc": worst_acc,
        "model_paths": best_model_paths,
        "train_log": train_log
    }, f, indent=4)

print(f"\n✅ Leave-One-Domain-Out TTA 結果已保存至 {result_path}")


---

### Cell 7: LODO 模型於所有 Domains 上的普通驗證（非 TTA）+ Resume 支援 + 結果視覺化

In [ ]:
import os
import json
import shutil
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# === 基本設定 ===
BASE_PACS_PATH = "/content/PACS"
RESULTS_DIR = "/content/drive/MyDrive/xAi_DG_TTA/lodo_tta_models"
RESUME_EVAL_PATH = os.path.join(RESULTS_DIR, "tta_eval_resume.json")
EVAL_RESULT_PATH = os.path.join(RESULTS_DIR, "lodo_tta_eval_results.json")
BEST_MODEL_COPY_DIR = os.path.join(RESULTS_DIR, "best_generalized_model")
os.makedirs(BEST_MODEL_COPY_DIR, exist_ok=True)

# === 儲存泛化結果 ===
eval_results = {}
all_models_avg_acc = {}

# === Resume 支援 ===
if os.path.exists(RESUME_EVAL_PATH):
    with open(RESUME_EVAL_PATH, "r") as f:
        eval_results = json.load(f)
    print("🔁 已載入之前的 TTA 評估結果，將繼續剩餘模型驗證...")

# === 驗證每個模型在所有 domain 上的泛化能力 ===
for target_domain in PACS_DOMAINS:
    if target_domain in eval_results:
        print(f"⏭️  已評估過 {target_domain}，略過...")
        continue

    print(f"\n===== Evaluating model trained on domain: {target_domain} =====")
    model_path = os.path.join(RESULTS_DIR, f"best_model_{target_domain}_tta.pth")
    if not os.path.exists(model_path):
        print(f"⚠️ 模型 {model_path} 不存在，跳過 {target_domain} 的驗證。")
        continue

    model = get_model(len(PACS_CATEGORIES), PRETRAINED).to(DEVICE)
    model.load_state_dict(torch.load(model_path))
    model.eval()

    domain_accs = {}
    for eval_domain in PACS_DOMAINS:
        val_dataset = PACS_Dataset(BASE_PACS_PATH, eval_domain, PACS_CATEGORIES, transform=val_test_transform)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

        val_loss, val_acc = eval_one_epoch(
            model, val_loader, torch.nn.CrossEntropyLoss(), DEVICE
        )
        domain_accs[eval_domain] = val_acc
        print(f"✔️ Accuracy on {eval_domain}: {val_acc:.3f}")

    avg_acc = np.mean(list(domain_accs.values()))
    all_models_avg_acc[target_domain] = avg_acc

    eval_results[target_domain] = {
        "model_path": model_path,
        "per_domain_acc": domain_accs,
        "avg_acc": avg_acc
    }

    # 寫入 resume 記錄
    with open(RESUME_EVAL_PATH, "w") as f:
        json.dump(eval_results, f, indent=4)

# === 儲存最終結果 JSON ===
with open(EVAL_RESULT_PATH, "w") as f:
    json.dump(eval_results, f, indent=4)
print(f"\n✅ 所有模型泛化結果已儲存至: {EVAL_RESULT_PATH}")

# === 找出泛化表現最好的模型 ===
if not all_models_avg_acc:
    print("❌ 沒有可用的模型驗證結果，請確認模型檔案是否命名正確。")
else:
    best_model_domain = max(all_models_avg_acc, key=all_models_avg_acc.get)
    best_model_path = eval_results[best_model_domain]["model_path"]
    best_model_copy_path = os.path.join(BEST_MODEL_COPY_DIR, f"best_model_generalized_tta.pth")
    shutil.copy2(best_model_path, best_model_copy_path)

    print(f"\n🏆 最佳泛化模型為: {best_model_domain}（平均 Acc: {all_models_avg_acc[best_model_domain]:.3f}）")
    print(f"📁 已將最佳模型複製至: {best_model_copy_path}")

    # === 畫圖：平均泛化準確率 ===
    plt.figure(figsize=(8, 5))
    domains = list(all_models_avg_acc.keys())
    accs = list(all_models_avg_acc.values())

    bars = plt.bar(domains, accs, color='skyblue')
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.01, f'{yval:.3f}', ha='center', va='bottom')

    avg_all = np.mean(accs)
    worst = np.min(accs)
    plt.axhline(avg_all, color='blue', linestyle='--', label='Average Accuracy')
    plt.axhline(worst, color='red', linestyle=':', label='Worst-case Accuracy')

    plt.title("Generalization Accuracy of LODO-TTA Models")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1.05)
    plt.legend()
    plt.tight_layout()
    plt.show()
